We're using the encoding="windows-1252" cause this dataset was exported from Windows Excel and containes special character.

In [72]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/superstore.csv", encoding="windows-1252")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [73]:
print(df.shape)
print()
print(df.dtypes)
print()
print(df.isna().sum())

(9994, 21)

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


1. Order ID and Order Date are both object, so python reads them as plain string texts - not dates. 
2. Column Names have space in them, and it's a nuance to use bracket notation with them
3. Sub-Category has a *hypen* in between, and it looks like a minus sign 
4. There's no missing value - the data is unusally clean 
5. Postal Code is int64, not a string. Postal codes aren't numbers.


# Renaming the columns 
spaces and hyphens to underscores, everything lowercase

In [74]:
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')
print(df.columns.tolist())

['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


# Fixing the date column

In [75]:
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=False)
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=False)

print(df['order_date'].dtype)
print(df['ship_date'].dtype)
print()
print(df[['order_date', 'ship_date']].head(3))

datetime64[ns]
datetime64[ns]

  order_date  ship_date
0 2016-11-08 2016-11-11
1 2016-11-08 2016-11-11
2 2016-06-12 2016-06-16


# Fix postal code
Changing Postal Codes to Strings. Two Reasons:
1. Postal Codes are identifiers, not quantities. Storing them as integers, implies one can do maths on them.
2. Some US postal codes starts with a "0". Pandas would read those as integers and drop the "0" in the beginning

In [76]:
df['postal_code'] = df['postal_code'].astype(str)

print(df['postal_code'].dtype)
print(df['postal_code'].head(5))

object
0    42420
1    42420
2    90036
3    33311
4    33311
Name: postal_code, dtype: object


In [77]:
short_codes = df[df['postal_code'].str.len() < 5]
print(f"Postal codes shorter than 5 digits: {len(short_codes)}")
print(short_codes['postal_code'].unique())

Postal codes shorter than 5 digits: 449
['6824' '7090' '7960' '7109' '8701' '7601' '1852' '6040' '2038' '2886'
 '1841' '7060' '7036' '8901' '3301' '6360' '2908' '6457' '2740' '8360'
 '8302' '2149' '7002' '6450' '6010' '7501' '1453' '8861' '3820' '2169'
 '5408' '4401' '2148' '1040' '7055' '7050' '2151' '2895' '2920' '6708'
 '4240' '6460' '3060' '2138' '1915' '8401' '7011' '1810' '6484' '6810'
 '1752' '7017']


# Fix postal code - "Data Corruption"

When pandas read the CSV with postal code stored as an integer, it dropped all the leading zeros.
So now, need to fix the postal codes, by inserting "0" in the beginning using **zfill**

In [78]:
df['postal_code'] = df['postal_code'].str.zfill(5)

# Verify
short_codes = df[df['postal_code'].str.len() < 5]
print(f"Postal codes shorter than 5 digits: {len(short_codes)}")
print(df['postal_code'].head(5))

Postal codes shorter than 5 digits: 0
0    42420
1    42420
2    90036
3    33311
4    33311
Name: postal_code, dtype: object


The isna().sum() showed that there are no missing values in the dataset. After performing, i'd like to check if there are still no missing values in the dataset. 
In addition, some missing values in the dataset are encoded as an empty string '' , 'None' or 'N/A' and pandas isna() would not catch that

In [79]:
print(pd.io.parsers.readers.STR_NA_VALUES)
print("=== NULL CHECK ===")
print(df.isna().sum())

print("\n=== EMPTY STRING CHECK (object columns only) ===")
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    empty_count = (df[col].str.strip() == '').sum()
    if empty_count > 0:
        print(f"{col}: {empty_count} empty strings")

print("\n=== SUSPICIOUS STRING CHECK ===")
suspicious = ['none', 'n/a', 'na', 'null', 'missing', 'unknown', '-']
for col in obj_cols:
    mask = df[col].str.strip().str.lower().isin(suspicious)
    count = mask.sum()
    if count > 0:
        print(f"{col}: {count} suspicious values")
print("Done")

{'', '#NA', 'nan', '1.#QNAN', 'NULL', '-1.#IND', 'NA', 'NaN', '1.#IND', '-1.#QNAN', 'n/a', '-NaN', '#N/A', '#N/A N/A', '-nan', '<NA>', 'N/A', 'None', 'null'}
=== NULL CHECK ===
row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

=== EMPTY STRING CHECK (object columns only) ===

=== SUSPICIOUS STRING CHECK ===
Done


# Check for duplicates

In [80]:
dupes = df.duplicated()
print(f"Duplicate rows: {dupes.sum()}")

Duplicate rows: 0


## Why SQLite?

At this point the data is clean and ready to analyse. I could continue using pandas 
for everything — for 9994 rows it would be fast and fine. But every BI/Analytics 
job asks for SQL, and my current portfolio doesn't show any.

In a real company, data lives in databases — Postgres, Snowflake, BigQuery. You 
query it with SQL. Nobody hands you a CSV. So rather than doing everything in pandas, 
I'm deliberately storing the data in a database and writing SQL queries to retrieve 
it — because that's how it works in practice.

I'm using SQLite specifically because it requires zero setup. No server, no 
installation, no password. The entire database is a single file (`data/superstore.db`). 
Anyone who clones this repo can run everything immediately without configuring anything.

From here on, every time I want data I write a SQL query. By the end of the project 
I'll have a set of real queries covering aggregations, filtering, grouping, and 
time-based analysis.

In [81]:
import sqlite3
from sqlalchemy import create_engine

# 1. Create the database file and engine
engine = create_engine('sqlite:///../data/superstore.db')

# 2. Write the dataframe to a table called 'orders'
df.to_sql('orders', con=engine, if_exists='replace', index=False)

# 3. Verify with a raw SQL query
with sqlite3.connect('../data/superstore.db') as conn:
    cursor = conn.execute("SELECT COUNT(*) FROM orders")
    print(f"Rows in database: {cursor.fetchone()[0]}")

Rows in database: 9994


In [82]:
def query(sql):
    with sqlite3.connect('../data/superstore.db') as conn:
        return pd.read_sql(sql, conn)

In [83]:
query("SELECT * FROM orders LIMIT 5")

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08 00:00:00.000000,2016-11-11 00:00:00.000000,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08 00:00:00.000000,2016-11-11 00:00:00.000000,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12 00:00:00.000000,2016-06-16 00:00:00.000000,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11 00:00:00.000000,2015-10-18 00:00:00.000000,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11 00:00:00.000000,2015-10-18 00:00:00.000000,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


# EDA 
1. What date range does the data cover?
2. What are the distinct regions, segments and categories?
3. How does the sales and profit looks overall - totals, averages, min/max?
4. Which year/month has the most orders?


In [84]:
query("""
    SELECT 
        MIN(order_date) AS first_order,
        MAX(order_date) AS last_order,
        COUNT(DISTINCT order_id) AS total_orders
    FROM orders
""")

,first_order,last_order,total_orders
0,2014-01-03 00:00:00.000000,2017-12-30 00:00:00.000000,5009


We have enough data for year-on-year comparision, monthly trends, and seasonal patterns


In [85]:
query("""
    SELECT DISTINCT region FROM orders ORDER BY region
""")

,region
0,Central
1,East
2,South
3,West


In [86]:
query("""
    SELECT DISTINCT segment FROM orders ORDER BY segment
""")

,segment
0,Consumer
1,Corporate
2,Home Office


In [87]:
query("""
    SELECT DISTINCT category FROM orders ORDER BY category
""")

,category
0,Furniture
1,Office Supplies
2,Technology


In [88]:
query("""
    SELECT DISTINCT sub_category FROM orders ORDER BY sub_category
""")

,sub_category
0,Accessories
1,Appliances
2,Art
3,Binders
4,Bookcases
5,Chairs
6,Copiers
7,Envelopes
8,Fasteners
9,Furnishings


In [89]:
query("""
    SELECT
        ROUND(SUM(sales), 2)        AS total_sales,
        ROUND(AVG(sales), 2)        AS avg_order_sales,
        ROUND(SUM(profit), 2)       AS total_profit,
        ROUND(AVG(profit), 2)       AS avg_order_profit,
        ROUND(SUM(profit) / SUM(sales) * 100, 2) AS profit_margin_pct
    FROM orders
""")

,total_sales,avg_order_sales,total_profit,avg_order_profit,profit_margin_pct
0,2297200.86,229.86,286397.02,28.66,12.47


## Overall Sales & Profit Summary

Querying the full dataset gives a first look at the business's financial health.

**Total sales: $2,297,200** across 4 years (2014–2017).  
**Total profit: $286,397** — a profit margin of **12.47%**.

That margin is worth pausing on. For every dollar of revenue, only 12.5 cents is 
profit. That's thin for a retail business — it suggests either heavy discounting, 
high costs in certain categories, or both. This is worth investigating by category 
and region later.

**Average sales per line item: $229.86** — one row in this dataset is one product 
within an order, not a full order. Since orders average ~2 line items, the average 
order value is closer to ~$460.

**Average profit per line item: $28.66** — consistent with the 12.47% margin 
(12.47% of $229.86 ≈ $28.66), which is a good internal sanity check.

In [90]:
df_year = query("""
    WITH yearly AS (
        SELECT
            STRFTIME('%Y', order_date)               AS year,
            ROUND(SUM(sales), 2)                     AS total_sales,
            ROUND(SUM(profit), 2)                    AS total_profit,
            ROUND(SUM(profit)/SUM(sales)*100, 2)     AS margin_pct,
            COUNT(DISTINCT order_id)                 AS num_orders
        FROM orders
        GROUP BY year
    )
    SELECT
        year,
        num_orders,
        total_sales,
        total_profit,
        margin_pct,
        ROUND((total_sales - LAG(total_sales) OVER (ORDER BY year)) 
              / LAG(total_sales) OVER (ORDER BY year) * 100, 1) AS sales_growth_pct,
        ROUND((total_profit - LAG(total_profit) OVER (ORDER BY year)) 
              / LAG(total_profit) OVER (ORDER BY year) * 100, 1) AS profit_growth_pct,
        ROUND(margin_pct - LAG(margin_pct) OVER (ORDER BY year), 2) AS margin_change_pp
    FROM yearly
    ORDER BY year
""")

df_year

,year,num_orders,total_sales,total_profit,margin_pct,sales_growth_pct,profit_growth_pct,margin_change_pp
0,2014,969,484247.50,49543.97,10.23,NaN,NaN,NaN
1,2015,1038,470532.51,61618.60,13.10,-2.8,24.4,2.87
2,2016,1315,609205.60,81795.17,13.43,29.5,32.7,0.33
3,2017,1687,733215.26,93439.27,12.74,20.4,14.2,-0.69


In [91]:
import plotly.graph_objects as go

DARK_BG = '#0f1117'
CARD_BG = '#1a1d27'
ACCENT = '#4f8ef7'
ACCENT2 = '#f7914f'
TEXT = '#e0e0e0'
GRID = '#2a2d3a'

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_year['year'],
    y=df_year['total_sales'],
    name='Total Sales',
    marker_color=ACCENT,
    opacity=0.85
))

fig.add_trace(go.Scatter(
    x=df_year['year'],
    y=df_year['margin_pct'],
    name='Profit Margin %',
    mode='lines+markers',
    line=dict(color=ACCENT2, width=2.5),
    marker=dict(size=8),
    yaxis='y2'
))

fig.update_layout(
    title=dict(text='Annual Sales & Profit Margin', pad=dict(l=20, t=20)),
    plot_bgcolor=DARK_BG,
    paper_bgcolor=DARK_BG,
    font=dict(color=TEXT),
    legend=dict(
        bgcolor=CARD_BG,
        orientation='h',
        yanchor='bottom',
        y=-0.2,
        xanchor='center',
        x=0.5
    ),
    yaxis=dict(title='Total Sales ($)', gridcolor=GRID, tickformat='$,.0f'),
    yaxis2=dict(title='Margin %', overlaying='y', side='right', gridcolor=GRID, ticksuffix='%'),
    xaxis=dict(gridcolor=GRID)
)

fig.show()

## Year-over-Year Growth

To compare each year against the previous one, I used a CTE to first calculate 
yearly totals, then applied a window function (`LAG()`) to fetch the previous 
year's values and compute percentage changes.

**Key findings from the chart:**

Margin increasing year by year from 2014 to 2016, but a slight drop in 2017 despite the increase in sales. 

**2015** — Even though the sales dropped from 2014, the margin increased in 2015 in comparison to 2014. 

**2016** — strongest year overall. Sales +29.5%, profit +32.7%, margin nudged up 
another +0.33pp. The most efficient year in the dataset.

**2017** — highest total sales, but the margin dropped from 2016 by 0.69 percentage points. 

**Note on margin changes:** margin differences are expressed in percentage points 
(pp), not as a percentage change. If margin goes from 10% to 13%, that's +3pp — 
not +30%.

**Sales grew consistently across all four years, but profitability didn't keep pace — margin improved from 2014 to 2016 then slipped back in 2017, suggesting the revenue growth in the final year came at a cost to efficiency**



In [92]:
df_month = query("""
    WITH monthly AS (
        SELECT
            STRFTIME('%Y', order_date)        AS year,
            STRFTIME('%m', order_date)        AS month,
            ROUND(SUM(sales), 2)              AS total_sales,
            ROUND(SUM(profit), 2)             AS total_profit,
            COUNT(DISTINCT order_id)          AS num_orders
        FROM orders
        GROUP BY year, month
    )
    SELECT
        year || '-' || month             AS year_month,
        year,
        month,
        total_sales,
        total_profit,
        num_orders,
        ROUND(total_profit / total_sales * 100, 2) AS margin_pct
    FROM monthly
    ORDER BY year, month
""")

df_month

,year_month,year,month,total_sales,total_profit,num_orders,margin_pct
0,2014-01,2014,01,14236.90,2450.19,32,17.21
1,2014-02,2014,02,4519.89,862.31,28,19.08
2,2014-03,2014,03,55691.01,498.73,71,0.90
3,2014-04,2014,04,28295.35,3488.84,66,12.33
4,2014-05,2014,05,23648.29,2738.71,69,11.58
5,2014-06,2014,06,34595.13,4976.52,66,14.39
6,2014-07,2014,07,33946.39,-841.48,65,-2.48
7,2014-08,2014,08,27909.47,5318.10,72,19.05
8,2014-09,2014,09,81777.35,8328.10,130,10.18
9,2014-10,2014,10,31453.39,3448.26,78,10.96


In [93]:
fig2 = go.Figure()

# Trace 1 — monthly sales line
fig2.add_trace(go.Scatter(
    x=df_month['year_month'],
    y=df_month['total_sales'],
    name='Total Sales',
    mode='lines+markers',
    line=dict(color=ACCENT, width=2),
    marker=dict(size=5)
))

# Trace 2 — margin line on second y-axis
fig2.add_trace(go.Scatter(
    x=df_month['year_month'],
    y=df_month['margin_pct'],
    name='Profit Margin %',
    mode='lines+markers',
    line=dict(color=ACCENT2, width=2),
    marker=dict(size=5),
    yaxis='y2'
))

fig2.update_layout(
    title=dict(text='Monthly Sales & Profit Margin', pad=dict(l=20, t=20)),
    plot_bgcolor=DARK_BG,
    paper_bgcolor=DARK_BG,
    font=dict(color=TEXT),
    legend=dict(
        bgcolor=CARD_BG,
        orientation='h',
        yanchor='bottom',
        y=-0.5,
        xanchor='center',
        x=0.5
    ),
    xaxis=dict(
        gridcolor=GRID,
        tickangle=45,
        tickmode='array',
        tickvals=df_month['year_month'][::3],
    ),
    yaxis=dict(title='Total Sales ($)', gridcolor=GRID, tickformat='$,.0f'),
    yaxis2=dict(title='Margin %', overlaying='y', side='right', gridcolor=GRID, ticksuffix='%')
)

fig2.show()

## Month-on-Month Analysis

1. Clear seasonality in the data. Increase in the sales in the month of september, then a soft decline in oct and then again an incline in nov-dec and finally sharp decline of sales in january. 
2. Margin shows no clear seasonal pattern - it fluctuates from month-to-month without any consistent relationship with total sales.
3. There's an overall upward trend in total sales

In [94]:
df_region = query("""
    SELECT
        region,
        COUNT(DISTINCT order_id)                    AS num_orders,
        ROUND(SUM(sales), 2)                        AS total_sales,
        ROUND(SUM(profit), 2)                       AS total_profit,
        ROUND(SUM(profit)/SUM(sales)*100, 2)        AS margin_pct
    FROM orders
    GROUP BY region
    ORDER BY total_sales DESC
""")

df_region

,region,num_orders,total_sales,total_profit,margin_pct
0,West,1611,725457.82,108418.45,14.94
1,East,1401,678781.24,91522.78,13.48
2,Central,1175,501239.89,39706.36,7.92
3,South,822,391721.91,46749.43,11.93


## Region Analysis 

1. Central's margin is roughly half the West's margin. It's bringing the total margin of the company down. **Possible Explanation inclues:** Higher Discount Rates, Higher Tax Rate, or it could be more competition that's forcing them to sell at a lower price. 
2. West is the strongest region with the highest sales and highest margin.
2. South has interestingly lower number of orders in comparison to Central, but it still has higher margin.

In [95]:
df_category = query("""
    SELECT
        category,
        COUNT(DISTINCT order_id)                    AS num_orders,
        ROUND(SUM(sales), 2)                        AS total_sales,
        ROUND(SUM(profit), 2)                       AS total_profit,
        ROUND(SUM(profit)/SUM(sales)*100, 2)        AS margin_pct
    FROM orders
    GROUP BY category
    ORDER BY total_sales DESC
""")

df_category

,category,num_orders,total_sales,total_profit,margin_pct
0,Technology,1544,836154.03,145454.95,17.40
1,Furniture,1764,741999.80,18451.27,2.49
2,Office Supplies,3742,719047.03,122490.80,17.04


## Categorical Analysis

1. Furniture has pretty high total sales, but only a margin of roughly 2.5% which is negligible in comparison to Technology and Office Supplies. This is worth investigating further - Why does Furniture have such a lower margin and what sub-categories of Furniture is driving these results. Is it even worth selling Furniture, or they are actually important for brand building. Furniture's near-zero margin is likely the main drag on the company's overall 12.47% margin.
2. Technology leads here in both total_sales and margin.
3. Office Supplies has the highest number of orders - 3742 orders but only $719k in sales means roughly $192 per order, compared to Technology's $541 per order. Office Supplies sells frequently but in small amounts. 

In [105]:
fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Sales & Margin by Region', 'Sales & Margin by Category')
)

fig3.add_trace(
    go.Bar(
        x=df_region['total_sales'],
        y=df_region['region'],
        orientation='h',
        name='Region Sales',
        marker_color=ACCENT,
        opacity=0.85,
        text=df_region['margin_pct'].apply(lambda x: f'Margin: {x}%'),
        textposition='inside'
    ),
    row=1, col=1
)

fig3.add_trace(
    go.Bar(
        x=df_category['total_sales'],
        y=df_category['category'],
        orientation='h',
        name='Category Sales',
        marker_color=ACCENT,
        opacity=0.85,
        text=df_category['margin_pct'].apply(lambda x: f'Margin: {x}%'),
        textposition='inside'
    ),
    row=1, col=2
)

fig3.update_layout(
    plot_bgcolor=DARK_BG,
    paper_bgcolor=DARK_BG,
    font=dict(color=TEXT),
    showlegend=False,
    title=dict(text='Regional & Category Performance', pad=dict(l=20, t=20)),
    autosize = True,
    height=400,
    margin=dict(l=150, r=50, t=80, b=50)
)

fig3.update_xaxes(gridcolor=GRID, tickformat='$,.0f')
fig3.update_yaxes(gridcolor=GRID)
fig3.update_yaxes(ticksuffix='  ')

fig3.show()

In [124]:
fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Margin by Region', 'Margin by Category')
)
df_region = df_region.sort_values('margin_pct', ascending=True)
df_category = df_category.sort_values('margin_pct', ascending=True)

# Region lollipop
fig4.add_trace(
    go.Scatter(
        x=df_region['margin_pct'],
        y=df_region['region'],
        mode='markers',
        marker=dict(color=ACCENT, size=14),
        name='Region Margin'
    ),
    row=1, col=1
)

# Stem lines for region
for i, row in df_region.iterrows():
    fig4.add_shape(
        type='line',
        x0=0, x1=row['margin_pct'],
        y0=row['region'], y1=row['region'],
        line=dict(color=ACCENT, width=2),
        row=1, col=1
    )

# Category lollipop
fig4.add_trace(
    go.Scatter(
        x=df_category['margin_pct'],
        y=df_category['category'],
        mode='markers',
        marker=dict(color=ACCENT2, size=14),
        name='Category Margin'
    ),
    row=1, col=2
)

# Stem lines for category
for i, row in df_category.iterrows():
    fig4.add_shape(
        type='line',
        x0=0, x1=row['margin_pct'],
        y0=row['category'], y1=row['category'],
        line=dict(color=ACCENT2, width=2),
        row=1, col=2
    )

fig4.update_layout(
    plot_bgcolor=DARK_BG,
    paper_bgcolor=DARK_BG,
    font=dict(color=TEXT),
    showlegend=False,
    title=dict(text='Margin by Region & Category', pad=dict(l=-10, t=40)),
    height=400,
    margin=dict(l=150, r=50, t=80, b=50)
)

fig4.update_xaxes(gridcolor=GRID, ticksuffix='%', range=[0, 20])
fig4.update_yaxes(gridcolor=GRID)

fig4.show()

# Discount Analysis

1. Average discount by category/sub-category — are some categories discounted more?
2. Relationship between discount and margin — does higher discount = lower margin?
3. At what discount threshold does profit turn negative?
4. Which sub-categories have the worst discount/margin combination?

In [133]:
df_discount = query("""
    SELECT
        category,
        sub_category,
        ROUND(AVG(discount) * 100, 1)      AS avg_discount_pct,
        ROUND(SUM(profit)/SUM(sales) * 100, 2)   AS avg_margin_pct,
        COUNT(*)                           AS num_rows
    FROM orders
    WHERE sales > 0
    GROUP BY category, sub_category
    ORDER BY avg_discount_pct DESC
""")

df_discount

,category,sub_category,avg_discount_pct,avg_margin_pct,num_rows
0,Office Supplies,Binders,37.2,14.86,1523
1,Technology,Machines,30.6,1.79,115
2,Furniture,Tables,26.1,-8.56,319
3,Furniture,Bookcases,21.1,-3.02,228
4,Furniture,Chairs,17.0,8.10,617
5,Office Supplies,Appliances,16.7,16.87,466
6,Technology,Copiers,16.2,37.20,68
7,Technology,Phones,15.5,13.49,889
8,Furniture,Furnishings,13.8,14.24,957
9,Office Supplies,Fasteners,8.2,31.40,217


## Discount Analysis

On the first look, there is a general trend - heavily discounted products tend to 
have lower or negative margins. However this relationship is not fully consistent, 
suggesting discount policy alone does not determine profitability.

**Binders** is the most discounted sub-category at 37.2%. Is 37.2%  an 
aggressive discount for a 14.86% margin. Is it worth discounting it that much?

**Tables and Bookcases** are the real problem sub-categories — both discounted above 
20% and both have negative margins (-8.56% and -3.02%). Are they loss leaders driving 
sales of more profitable Furniture sub-categories? Worth investigating with a *basket 
analysis*.

**Supplies** has a negative margin of -2.55% on only 190 transactions. Too small to 
be a meaningful loss leader — likely just poor pricing on a low-volume product.

**Copiers** is an interesting anomaly - 16.2% discount but 37.2% margin. High-value 
products can absorb discounting in a way low-margin products cannot. The business 
should arguably be pushing Copiers harder.

**Chairs** is worth flagging — highest volume Furniture sub-category (617 rows) but 
only 8.1% margin at a 17% discount. Not loss-making, but barely contributing.

**The discount-margin relationship also breaks down at low discount levels** — Storage 
(7.5% discount, 9.51% margin) and Supplies (7.7% discount, -2.55% margin) suggest 
some products have structural cost problems unrelated to discounting.

A useful framework here is a **profitability matrix**

## Break-even Discount Threshold 

So far we've seen that heavily discounted products tend to have lower margins. The 
goal here is to find the actual discount level at which the business stops making 
money — so we can set a data-driven threshold instead of guessing.

To do this, we group all orders by their discount level and calculate the overall 
margin at each level. The point where margin crosses zero is our break-even threshold.

In [135]:
df_threshold = query("""
    SELECT
        ROUND(discount * 100, 0)                    AS discount_bucket,
        ROUND(SUM(profit)/SUM(sales) * 100, 2)      AS margin_pct,
        COUNT(*)                                     AS num_rows
    FROM orders
    WHERE sales > 0
    GROUP BY discount_bucket
    ORDER BY discount_bucket
""")

df_threshold

,discount_bucket,margin_pct,num_rows
0,0.0,29.51,4798
1,10.0,16.61,94
2,15.0,5.15,52
3,20.0,11.82,3657
4,30.0,-10.05,227
5,32.0,-16.50,27
6,40.0,-19.81,206
7,45.0,-45.45,11
8,50.0,-34.80,66
9,60.0,-89.46,138


## Break-even Analysis

1. The margin turns negative somewhere between 20% and 30%, estimated at around 25%. 
This becomes our data-driven threshold for the profitability matrix. 

2. At exactly 20% discount, there are almost 3700 products being sold — more than 
any other discount level except 0%. This is unlikely to be a coincidence.

3. Around 850 products are being sold at a discount level above 60%, driving very 
large negative margins — as low as -180% at 80% discount. It is worth investigating 
further what is driving these extreme discounts and whether they are authorised or 
accidental.

4. 4798 rows have 0% discount and a 29.51% margin. That's nearly half the dataset and the healthiest margin by far. This is actually the strongest finding in the whole table - when the business doesn't discount, it makes nearly 30% margin.

In [148]:
fig5 = go.Figure()

fig5.add_trace(go.Scatter(
    x=df_discount['avg_discount_pct'],
    y=df_discount['avg_margin_pct'],
    mode='markers',
    marker=dict(color=ACCENT, size=10),
    text=df_discount['sub_category'],
    hovertemplate='<b>%{text}</b><br>Discount: %{x}%<br>Margin: %{y}%<extra></extra>'
))

fig5.add_hline(y=0, line=dict(color='white', width=1, dash='dash'))
fig5.add_vline(x=25, line=dict(color=ACCENT2, width=1, dash='dash'))

fig5.update_layout(
    title=dict(text='Profitability Matrix — Discount vs Margin', pad=dict(l=20, t=20)),
    plot_bgcolor=DARK_BG,
    paper_bgcolor=DARK_BG,
    font=dict(color=TEXT),
    xaxis=dict(title='Average Discount %', gridcolor=GRID, ticksuffix='%'),
    yaxis=dict(title='Profit Margin %', gridcolor=GRID, ticksuffix='%'),
    height=500
)

fig5.add_annotation(x=3, y=47, text="⭐ Star Products",
    showarrow=False, font=dict(color='#aaaaaa', size=11))

fig5.add_annotation(x=3, y=6, text="📉 Underperformers",
    showarrow=False, font=dict(color='#aaaaaa', size=11))

fig5.add_annotation(x=3, y=-13, text="⚠️ Structural Problems",
    showarrow=False, font=dict(color='#aaaaaa', size=11))

fig5.add_annotation(x=27, y=47, text="💪 Resilient Products",
    showarrow=False, font=dict(color='#aaaaaa', size=11))

fig5.add_annotation(x=27.7, y=6, text="⚡ Over-discounted, Surviving",
    showarrow=False, font=dict(color='#aaaaaa', size=11))

fig5.add_annotation(x=27, y=-13, text="🚨 Problem Products",
    showarrow=False, font=dict(color='#aaaaaa', size=11))

fig5.add_hline(y=12.47, line=dict(color=ACCENT2, width=1, dash='dot'),
               annotation_text='Company avg margin (12.47%)',
               annotation_position='right',
               annotation_font=dict(color=ACCENT2, size=10))

fig5.show()

In [153]:
df_disc_vol_sub = query("""
    SELECT
        sub_category,
        ROUND(discount * 100, 0)        AS discount_bucket,
        ROUND(SUM(profit)/SUM(sales)*100, 2) AS margin_pct,
        ROUND(AVG(quantity), 2)         AS avg_quantity,
        COUNT(*)                        AS num_rows
    FROM orders
    WHERE sub_category IN ('Tables', 'Bookcases', 'Binders', 'Machines')
    AND sales > 0
    GROUP BY sub_category, discount_bucket
    ORDER BY sub_category, discount_bucket
""")

df_disc_vol_sub

,sub_category,discount_bucket,margin_pct,avg_quantity,num_rows
0,Binders,0.0,48.04,3.83,337
1,Binders,20.0,34.43,3.89,573
2,Binders,70.0,-73.59,3.96,380
3,Binders,80.0,-161.32,4.09,233
4,Bookcases,0.0,19.02,3.52,60
5,Bookcases,15.0,5.15,3.81,52
6,Bookcases,20.0,0.49,3.63,46
7,Bookcases,30.0,-12.98,4.30,10
8,Bookcases,32.0,-16.50,3.89,27
9,Bookcases,50.0,-58.23,4.22,18


## Discount Analysis — Final Conclusions

The core finding across all analysis is simple: **discounting is not driving volume**. 
Average quantity per order stays almost flat regardless of discount level. The business 
is giving away margin without getting any meaningful increase in units sold in return.

**Binders** — 48.04% margin at 0% discount, currently being sold at 37.2% average 
discount. Volume barely changes across discount levels. This product does not need 
discounting at all — reducing or eliminating the discount would significantly improve 
profitability with minimal volume risk.

**Bookcases** — 19.02% margin at 0% discount, turns negative above 20% discount. 
Volume does not meaningfully increase with higher discounts. Recommendation: stop 
discounting above 15%.

**Machines** — 38.20% margin at 0% discount, one of the best products in the dataset 
when sold at full price. Currently being discounted heavily. Recommendation: 
Investigate the root cause of heavy Machines discounting.

**Tables** — 18.55% margin at 0% discount, loss-making at every discount level above 
20%. Like the others, volume does not justify the discount. Recommendation: sell at 
full price or with a maximum 15% discount.

**The big picture:** all four problem sub-categories are actually healthy products when 
sold at full price. None of them need to be discontinued — the discounting is entirely 
a policy problem. If the business capped discounts at 20% across these four products, 
it would recover significant margin without meaningful volume loss.

In [179]:
df_top_customers = query("""
    SELECT
        customer_name,
        COUNT(DISTINCT order_id)                    AS num_orders,
        ROUND(SUM(sales), 2)                        AS total_sales,
        ROUND(SUM(profit), 2)                       AS total_profit,
        ROUND(SUM(profit)/SUM(sales)*100, 2)        AS margin_pct
    FROM orders
    GROUP BY customer_name
    ORDER BY total_profit DESC
    LIMIT 5
""")

df_top_customers

,customer_name,num_orders,total_sales,total_profit,margin_pct
0,Tamara Chand,5,19052.22,8981.32,47.14
1,Raymond Buch,6,15117.34,6976.10,46.15
2,Sanjit Chand,9,14142.33,5757.41,40.71
3,Hunter Lopez,6,12873.30,5622.43,43.68
4,Adrian Barton,10,14473.57,5444.81,37.62


In [180]:
df_worst_customers = query("""
    SELECT
        customer_name,
        COUNT(DISTINCT order_id)                    AS num_orders,
        ROUND(SUM(sales), 2)                        AS total_sales,
        ROUND(SUM(profit), 2)                       AS total_profit,
        ROUND(SUM(profit)/SUM(sales)*100, 2)        AS margin_pct
    FROM orders
    GROUP BY customer_name
    ORDER BY total_profit 
    LIMIT 5
""")

df_worst_customers

,customer_name,num_orders,total_sales,total_profit,margin_pct
0,Cindy Stewart,6,5690.05,-6626.39,-116.46
1,Grant Thornton,3,9351.21,-4108.66,-43.94
2,Luke Foster,7,3930.51,-3583.98,-91.18
3,Sharelle Roach,5,3233.48,-3333.91,-103.11
4,Henry Goldwyn,12,3247.64,-2797.96,-86.15


In [194]:
fig6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top 5 Customers by Profit', 'Bottom 5 Customers by Profit')
)

# Top 5 — sorted ascending so highest is at top
df_top5_sorted = df_top_customers.sort_values('total_profit', ascending=True)

fig6.add_trace(
    go.Bar(
        x=df_top5_sorted['total_profit'],
        y=df_top5_sorted['customer_name'],
        orientation='h',
        marker_color=ACCENT,
        opacity=0.85,
        name='Top 5',
        text=df_top5_sorted['margin_pct'].apply(lambda x: f'Margin: {x}%'),
        textposition='inside',
        textfont=dict(color='white', size=10)
    ),
    row=1, col=1
)

# Bottom 5 — sorted descending so most negative is at bottom
df_bottom5_sorted = df_worst_customers.sort_values('total_profit', ascending=False)

fig6.add_trace(
    go.Bar(
        x=df_bottom5_sorted['total_profit'],
        y=df_bottom5_sorted['customer_name'],
        orientation='h',
        marker_color='#e05c5c',
        opacity=0.85,
        name='Bottom 5',
        text=df_bottom5_sorted['margin_pct'].apply(lambda x: f'Margin: {x}%'),
        textposition='inside',
        textfont=dict(color='white', size=10)
    ),
    row=1, col=2
)

fig6.update_layout(
    plot_bgcolor=DARK_BG,
    paper_bgcolor=DARK_BG,
    font=dict(color=TEXT),
    showlegend=False,
    title=dict(text='Customer Profitability — Best & Worst', pad=dict(l=20, t=20)),
    height=400,
    margin=dict(l=150, r=50, t=80, b=50)
)

fig6.update_xaxes(gridcolor=GRID, tickformat='$,.0f')
fig6.update_yaxes(gridcolor=GRID)
fig6.update_yaxes(gridcolor=GRID, ticksuffix='  ')

fig6.show()

## Customer Profitability — Best & Worst

**Top 5 customers** are the most profitable customers in the dataset, all with margins 
of roughly 40% or above. These are the customers the business should prioritise 
retaining and growing.

**Bottom 5 customers** are actively losing the business money. Cindy Stewart alone 
has a margin of -116%, meaning the business is losing more than it makes on every 
sale to her. The key questions are: what products are we selling them, are we 
discounting heavily for them, and do we need to renegotiate terms?

**The contrast is striking** — Cindy Stewart is losing the business $6,626 while 
Tamara Chand, the best customer, brings in $8,981. The losses from bad customers 
are nearly as large as the gains from the best ones.

In [195]:
query("""
    SELECT
        category,
        sub_category,
        COUNT(*)                                    AS num_rows,
        ROUND(SUM(sales), 2)                        AS total_sales,
        ROUND(SUM(profit), 2)                       AS total_profit,
        ROUND(SUM(profit)/SUM(sales)*100, 2)        AS margin_pct,
        ROUND(AVG(discount)*100, 1)                 AS avg_discount_pct
    FROM orders
    WHERE customer_name = 'Cindy Stewart'
    GROUP BY category, sub_category
    ORDER BY total_profit ASC
""")

,category,sub_category,num_rows,total_sales,total_profit,margin_pct,avg_discount_pct
0,Technology,Machines,2,4895.98,-6409.90,-130.92,35.0
1,Office Supplies,Binders,1,456.59,-304.39,-66.67,70.0
2,Office Supplies,Supplies,1,52.14,5.87,11.25,20.0
3,Office Supplies,Labels,1,18.75,9.00,48.00,0.0
4,Technology,Accessories,1,59.98,12.00,20.00,20.0
5,Office Supplies,Paper,2,38.52,17.33,45.00,0.0
6,Office Supplies,Appliances,1,168.10,43.71,26.00,0.0


In [196]:
query("""
    SELECT
        category,
        sub_category,
        COUNT(*)                                    AS num_rows,
        ROUND(SUM(sales), 2)                        AS total_sales,
        ROUND(SUM(profit), 2)                       AS total_profit,
        ROUND(SUM(profit)/SUM(sales)*100, 2)        AS margin_pct,
        ROUND(AVG(discount)*100, 1)                 AS avg_discount_pct
    FROM orders
    WHERE customer_name = 'Sharelle Roach'
    GROUP BY category, sub_category
    ORDER BY total_profit ASC
""")

,category,sub_category,num_rows,total_sales,total_profit,margin_pct,avg_discount_pct
0,Technology,Machines,1,2549.99,-3399.98,-133.33,70.0
1,Office Supplies,Binders,3,42.59,-31.63,-74.27,70.0
2,Furniture,Chairs,2,297.33,-0.02,-0.01,15.0
3,Office Supplies,Paper,1,20.74,7.26,35.00,20.0
4,Office Supplies,Art,1,109.90,32.97,30.00,0.0
5,Furniture,Bookcases,1,212.94,57.49,27.00,0.0


In [197]:
query("""
    SELECT
        category,
        sub_category,
        COUNT(*)                                    AS num_rows,
        ROUND(SUM(sales), 2)                        AS total_sales,
        ROUND(SUM(profit), 2)                       AS total_profit,
        ROUND(SUM(profit)/SUM(sales)*100, 2)        AS margin_pct,
        ROUND(AVG(discount)*100, 1)                 AS avg_discount_pct
    FROM orders
    WHERE customer_name = 'Tamara Chand'
    GROUP BY category, sub_category
    ORDER BY total_profit ASC
""")

,category,sub_category,num_rows,total_sales,total_profit,margin_pct,avg_discount_pct
0,Office Supplies,Storage,1,32.48,4.87,15.00,0.0
1,Office Supplies,Art,1,33.96,9.51,28.00,0.0
2,Office Supplies,Envelopes,1,74.35,26.95,36.25,20.0
3,Office Supplies,Paper,2,85.32,40.58,47.56,0.0
4,Technology,Accessories,1,498.00,184.26,37.00,0.0
5,Office Supplies,Binders,5,828.16,315.18,38.06,24.0
6,Technology,Copiers,1,17499.95,8399.98,48.00,0.0


## Customer vs Discount Analysis

Investigating individual customer purchases reveals that the product is rarely the 
problem — the discount applied to it is.

**Tamara Chand** (best customer) was sold Binders at 24% discount with a 38% margin. 
Her single Copiers transaction at 0% discount generated $8,399 profit — almost her 
entire contribution to the business.

**Cindy Stewart** (worst customer) was sold the same Binders at 70% discount, 
generating a -67% margin. Her Machines purchase at 35% discount lost the business 
$6,409 in a single transaction.

Same products, same business, completely different outcomes — purely because of the 
discount applied.

This reframes the customer profitability problem. It is not a customer problem or a 
product problem. It is a **sales governance problem**. The business needs discount 
approval thresholds — extreme discounts like 70% should require management sign-off 
before being applied.

**Questions worth raising with management:**
1. Who is authorising these heavily discounted sales and why?
2. Is there a maximum discount policy in place, and if so, is it being enforced?

## Forecasting

Now we'll start forecasting the data and predict future sales using "Prophet".
The reasons I chose prophet here:
1. open-source library especially designed for time-series data
2. It handles seasonality automatically and the data shows clear seasonalilty 

Goal:
1. Train the model on the existing dataset (2014 - 2017)
2. Forecast the next 12 months

"Multiplicative seasonality" was chosen because as overall sales grow, the seasonal spikes also grow proportionally — which matches the pattern we observed in the monthly analysis.

In [200]:
from prophet import Prophet
import pandas as pd

# Prepare data for Prophet
df_prophet = query("""
    SELECT
        DATE(order_date, 'start of month')  AS ds,
        ROUND(SUM(sales), 2)                AS y
    FROM orders
    GROUP BY ds
    ORDER BY ds
""")

# Convert ds to datetime
df_prophet['ds'] = pd.to_datetime(df_prophet['ds'])

print(df_prophet.shape)
df_prophet.head()

(48, 2)


,ds,y
0,2014-01-01,14236.90
1,2014-02-01,4519.89
2,2014-03-01,55691.01
3,2014-04-01,28295.35
4,2014-05-01,23648.29


In [201]:
# Train Prophet model
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative'
)

model.fit(df_prophet)

# Create future dataframe — 12 months ahead
future = model.make_future_dataframe(periods=12, freq='MS')

# Generate forecast
forecast = model.predict(future)

# Show last 6 rows — the future predictions
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

11:54:22 - cmdstanpy - INFO - Chain [1] start processing
11:54:25 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
48,2018-01-01,36566.830679,26779.586217,46461.295645
49,2018-02-01,19564.353615,8682.491720,29495.110422
50,2018-03-01,70738.825919,61575.380447,80609.192296
51,2018-04-01,48739.241026,38776.779159,58733.670346
52,2018-05-01,48380.635737,39101.348497,59670.789272
53,2018-06-01,50403.587888,40945.099207,59701.792321
54,2018-07-01,49784.944146,39523.427789,59702.006879
55,2018-08-01,57547.002547,47629.810178,67507.778237
56,2018-09-01,105865.323358,96535.061552,115582.499325
57,2018-10-01,62413.890667,52842.347921,72680.927780


In [210]:
fig7 = go.Figure()

# Confidence interval
fig7.add_trace(go.Scatter(
    x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
    y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(79, 142, 247, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Confidence Interval'
))

# Actual sales
fig7.add_trace(go.Scatter(
    x=df_prophet['ds'],
    y=df_prophet['y'],
    mode='lines+markers',
    name='Actual Sales',
    line=dict(color=ACCENT2, width=2),
    marker=dict(size=5)
))

# Forecast
fig7.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat'],
    mode='lines',
    name='Forecast',
    line=dict(color=ACCENT, width=2, dash='dash')
))

# Vertical line at forecast start
fig7.add_vline(
    x=pd.Timestamp('2018-01-01').timestamp() * 1000,
    line=dict(color='white', width=1, dash='dot'),
    annotation_text='Forecast Start',
    annotation_position='top right',
    annotation_font=dict(color='white', size=10)
)

fig7.update_layout(
    title=dict(text='Sales Forecast — Prophet Model (2014–2018)', pad=dict(l=20, t=20)),
    plot_bgcolor=DARK_BG,
    paper_bgcolor=DARK_BG,
    font=dict(color=TEXT),
    legend=dict(
        bgcolor=CARD_BG,
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5
    ),
    xaxis=dict(title='Date', gridcolor=GRID),
    yaxis=dict(title='Sales ($)', gridcolor=GRID, tickformat='$,.0f'),
    height=500
)

fig7.show()

## Forecast Analysis

1. As we can see from the graph, the forecast is following along closely with the actual sales from 2014 till 2017 end. So we can say the model predicts well and as we can see as it forecast from 2018 onwards, it follows the seasonality pattern that we have been seeing. 
2. The total sales seems to be increasing in the year 2018, which is a good sign. We still have to see how much profit margin, we could expect, can we expect it to increase -- that would be our next goal (Future Goal)
3. Issue with Prophet: It extrapolates patterns it has seen — it can't account for external shocks like a new competitor, economic downturn, or supply chain issues. 
4. The shaded band shows the confidence interval — the range within which actual sales are likely to fall.  It remains relatively tight throughout because Prophet is confident in the seasonal pattern after observing it repeat across 4 years.